# Cox Integration

In [1]:
import time
import pickle
import json
from lifelines import CoxPHFitter
import copy
import os
import torch

## model auditing
from pathlib import Path
import numpy as np
import torch.utils.data
from sklearn.metrics import roc_curve, auc
from torch.utils.data import Subset
from attacks import tune_offline_a, run_rmia, run_loss
from visualize import plot_roc, plot_roc_log

## build model and dataset
import pandas as pd
from lifelines.datasets import load_rossi
import matplotlib.pyplot as plt

## synthetic dataset
import glob

/home/vesper/.local/lib/python3.12/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (
/home/vesper/.local/lib/python3.12/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


In [2]:
global_log_dir = "heterogeneity/"

## split dataset for training

In [3]:
# whole dataset
def split_dataframe_for_training(dataframe, num_model_pairs):
    """
    Split a Pandas DataFrame into training and test partitions for model pairs.

    Args:
        dataframe (pd.DataFrame): Input dataset as a Pandas DataFrame.
        num_model_pairs (int): Number of model pairs to be trained, with each pair trained on different halves of the dataset.

    Returns:
        data_splits (list): List of dictionaries containing training and test DataFrames for each model.
        master_keep (np.array): Boolean array indicating the membership of samples in each model's training set.
    """
    dataset_size = len(dataframe)
    indices = np.arange(dataset_size)
    split_index = dataset_size // 2
    master_keep = np.full((2 * num_model_pairs, dataset_size), True, dtype=bool)
    data_splits = []
    
    training_indices = []

    for i in range(num_model_pairs):
        # Shuffle indices to randomize the dataset
        np.random.shuffle(indices)
        
        #print(indices)
        
        # Update master_keep for training and testing sets
        master_keep[i * 2, indices[split_index:]] = False
        master_keep[i * 2 + 1, indices[:split_index]] = False

        # record training indices
        training_indices.append(indices[:split_index].copy())
        training_indices.append(indices[split_index:].copy())
        
        # Generate train and test indices
        train_indices_1 = np.where(master_keep[i * 2, :])[0]
        test_indices_1 = np.where(~master_keep[i * 2, :])[0]

        train_indices_2 = np.where(master_keep[i * 2 + 1, :])[0]
        test_indices_2 = np.where(~master_keep[i * 2 + 1, :])[0]
        
        # Append training and testing DataFrames for both models in the pair
        data_splits.append(
            {
                "train": dataframe.iloc[train_indices_1],
                "test": dataframe.iloc[test_indices_1],
            }
        )
        data_splits.append(
            {
                "train": dataframe.iloc[train_indices_2],
                "test": dataframe.iloc[test_indices_2],
            }
        )

    return data_splits, master_keep, training_indices

In [4]:
# non-censored dataset
def split_dataframe_for_training(dataframe, num_model_pairs):
    """
    Split a Pandas DataFrame into training and test partitions, ensuring each 
    model's training set contains exactly half of the non-censored records 
    (where 'arrest' == 1).

    Args:
        dataframe (pd.DataFrame): Input dataset as a Pandas DataFrame.
        num_model_pairs (int): Number of model pairs to be trained.

    Returns:
        data_splits (list): List of dictionaries containing training and 
                           test DataFrames for each model.
        master_keep (np.array): Boolean array indicating sample membership.
    """

    # Identify non-censored records (arrest == 1)
    non_censored_indices = dataframe[dataframe['arrest'] == 1].index.to_numpy()
    num_non_censored = len(non_censored_indices)

    mp = {}
    val = 0
    for idx in non_censored_indices:
        mp[idx] = val
        val += 1
    
    # Handle censored records (arrest == 0):
    censored_indices = dataframe[dataframe['arrest'] == 0].index.to_numpy()
    num_censored = len(censored_indices)

    dataset_size = len(dataframe)
    indices = np.arange(dataset_size)
    master_keep = np.full((2 * num_model_pairs, len(non_censored_indices)), False, dtype=bool)
    data_splits = []

    for i in range(num_model_pairs):
        np.random.shuffle(censored_indices)
        np.random.shuffle(non_censored_indices)
        
        # non censored
        split_index_non_censored = num_non_censored // 2

        train_non_censored_indices_1 = non_censored_indices[:split_index_non_censored]
        test_non_censored_indices_1 = non_censored_indices[split_index_non_censored:]

        train_non_censored_indices_2 = non_censored_indices[split_index_non_censored:]
        test_non_censored_indices_2 = non_censored_indices[:split_index_non_censored]
        
        # censored
        split_index_censored = len(censored_indices) // 2

        train_censored_indices_1 = censored_indices[:split_index_censored]
        test_censored_indices_1 = censored_indices[split_index_censored:]

        train_censored_indices_2 = censored_indices[split_index_censored:]
        test_censored_indices_2 = censored_indices[:split_index_censored]

        # Combine indices for training and testing
        train_indices_1 = np.concatenate([train_non_censored_indices_1, train_censored_indices_1])
        test_indices_1 = np.concatenate([test_non_censored_indices_1, test_censored_indices_1])split_dataframe_for_training

        train_indices_2 = np.concatenate([train_non_censored_indices_2, train_censored_indices_2])
        test_indices_2 = np.concatenate([test_non_censored_indices_2, test_censored_indices_2])

        '''
        # Update master_keep (membership)
        master_keep[i * 2, train_indices_1] = True
        master_keep[i * 2, test_indices_1] = False

        master_keep[i * 2 + 1, train_indices_2] = True
        master_keep[i * 2 + 1, test_indices_2] = False
        '''


        for idx in train_non_censored_indices_1:
            master_keep[i * 2, mp[idx]] = True
        for idx in train_non_censored_indices_2:
            master_keep[i * 2 + 1, mp[idx]] = True


        data_splits.append(
            {
                "train": dataframe.iloc[train_indices_1],
                "test": dataframe.iloc[test_indices_1],
            }
        )
        data_splits.append(
            {
                "train": dataframe.iloc[train_indices_2],
                "test": dataframe.iloc[test_indices_2],
            }
        )

    return data_splits, master_keep

## train cox models using the splitted dataset

In [4]:
def train_cox_models(data_splits, num_model_pairs, log_dir):
    """
    Train Cox proportional hazards models using data splits and store metadata.

    Args:
        data_splits (list): List of dictionaries containing "train" and "test" DataFrames.
        num_model_pairs (int): Number of model pairs to train.
        log_dir (str): Directory to store models and metadata.
        logger (logging.Logger): Logger for logging information.
        dataset_name (str): Name of the dataset used for training.

    Returns:
        list: List of trained CoxPHFitter models.
    """
    os.makedirs(log_dir, exist_ok=True)  # Ensure log directory exists
    model_list = []
    model_metadata_dict = {}

    for split_idx, split_info in enumerate(data_splits):
        #print(f"Training model {split_idx}...")

        # Extract train and test sets
        train_data = split_info["train"]
        test_data = split_info["test"]

        # Initialize the CoxPH model
        model = CoxPHFitter()
        baseline_time = time.time()

        # Train the Cox model
        model.fit(train_data, duration_col='time', event_col='event')

        # Evaluate the model
        train_score = model.score(train_data, scoring_method="concordance_index")
        test_score = model.score(test_data, scoring_method="concordance_index")

        #train_loss = model.log_likelihood_ratio_
        #test_loss = model.compute_residuals(test_data, "martingale").abs().mean()

       # logger.info(
       #     "Training model %s took %.2f seconds", split_idx, time.time() - baseline_time
       # )

        # Store the trained model
        model_list.append(copy.deepcopy(model))

        model_idx = split_idx

        with open(f"{log_dir}/model_{model_idx}.pkl", "wb") as f:
            pickle.dump(model, f)

        # Store metadata
        model_metadata_dict[model_idx] = {
            "model_name": "CoxPHFitter",
            "num_train": len(train_data),
            "num_test": len(test_data),
            "train_acc": train_score,
            "test_acc": test_score,
            #"train_loss": train_loss,
            #"test_loss": test_loss,
            #"dataset": dataset_name,
            "model_path": f"{log_dir}/cox_model_{model_idx}.pkl",
        }

    # Save all metadata as a JSON file
    with open(f"{log_dir}/cox_models_metadata.json", "w") as f:
        json.dump(model_metadata_dict, f, indent=4)

    return model_list, model_metadata_dict


## prepare auditing dataset and memberships

In [10]:
# use original dataset and memberships (if no downsamples specified)
def sample_cox_auditing_dataset(dataset, memberships):
    return dataset, memberships

In [7]:
# auditing dataset only covers non-censored dataset
def sample_cox_auditing_dataset(dataset: pd.DataFrame, memberships) -> pd.DataFrame:
    """
    Extracts all non-censored records from the dataset where the 'arrest' column is 1.

    Parameters:
    dataset (pd.DataFrame): The input dataset (Rossi dataset).

    Returns:
    pd.DataFrame: A new dataset containing only non-censored records.
    """
    return dataset[dataset['arrest'] == 1].copy(), memberships

## Compute Attack Signals

## compute model signals
- Use "partial likelihood per sample" to generate pseudo "loss" values. use get_loss function to compute signals and do attacks.

- Function: 

def cox_partial_likelihood_per_sample(predicted_risks, true_events, true_times)

- Explaination:

  """
  Calculates the Cox partial likelihood loss for each sample.

  Args:
      predicted_risks (torch.Tensor): Predicted risks from the Cox model.
  
      true_events (torch.Tensor): Indicator vector for events (1 for event, 0 for censored).
  
      true_times (torch.Tensor): Survival times for each sample.

  Returns:
      torch.Tensor: A tensor containing the loss for each sample.
  """

### compute attack signals based on square of distance

In [5]:
# one dataset for one model

def get_cox_model_signals(model_list, data_list, log_dir=global_log_dir):
    print("calculating attack signals using square of distance...")
    signals = []
    model_idx = 0
    
    for model in model_list:
        print(f"Compute attack signals for model {model_idx}...")
        
        predicted_times = torch.tensor(model.predict_expectation(data_list[model_idx]).values) 
        true_events = torch.tensor(data_list[model_idx]['event'].values)
        true_times = torch.tensor(data_list[model_idx]['time'].values)

        model_idx += 1

        n = len(predicted_times)
        sample_losses = np.zeros(n)
        
        for i in range(n):
            if true_events[i] == 1:
                if predicted_times[i] == float('inf'):
                    sample_losses[i] = float('inf')
                else:
                    sample_losses[i] = (predicted_times[i] - true_times[i]) * (predicted_times[i] - true_times[i])
            else:
                if predicted_times[i] == float('inf') or predicted_times[i] >= true_times[i]:
                    sample_losses[i] = 0
                else:
                    # option 1: underestimate signals
                    #sample_losses[i] = (predicted_times[i] - true_times[i]) * (predicted_times[i] - true_times[i])
                    
                    # option 2: manually make signals larger
                    correction_value = 100
                    sample_losses[i] = (true_times[i] - predicted_times[i] + correction_value) * (true_times[i] - predicted_times[i] + correction_value)
                    
                    # option 3: cannot deal with infinite, must be postprocessed
                    #sample_losses[i] = 256 #float('inf')
                    
        signals.append(sample_losses.reshape(-1, 1))
        
    signals = np.concatenate(signals, axis = 1)
    np.save(
        f"{log_dir}/cox_square_signals.npy",
        signals,
    )
    print("Signals saved to disk.")
    return signals

### compute loss based on individual c-index

In [10]:
# performance optimization using vectorization

def get_cox_model_signals(model_list, data, log_dir=global_log_dir):
    signals = []
    model_idx = 0

    print(f"Total: {len(data)} signals to be calculated...")
    
    for model in model_list:
        print(f"Compute attack signals for model {model_idx}...")
        model_idx += 1
        
        # Convert data to PyTorch tensors
        predicted_times = torch.tensor(model.predict_expectation(data).values)  
        true_events = torch.tensor(data['event'].values)
        true_times = torch.tensor(data['time'].values)
        
        # Get total number of samples
        n = len(predicted_times)
        
        # Expand dimensions to enable broadcasting
        true_times_i = true_times.view(n, 1)
        true_times_j = true_times.view(1, n)
        predicted_times_i = predicted_times.view(n, 1)
        predicted_times_j = predicted_times.view(1, n)
        true_events_i = true_events.view(n, 1)
        true_events_j = true_events.view(1, n)
        
        # Compute masks for different comparison cases
        non_censored_i = true_events_i == 1
        non_censored_j = true_events_j == 1
        censored_i = ~non_censored_i
        
        # Condition 1: Both are non-censored
        valid_pairs_1 = non_censored_i & non_censored_j
        correct_pairs_1 = ((true_times_i > true_times_j) == (predicted_times_i > predicted_times_j)).float()
        correct_pairs_1 += ((true_times_i == true_times_j) & (predicted_times_i == predicted_times_j)).float() * 0.5
        
        # Condition 2: i is non-censored, j is censored & j's time is >= i's time
        valid_pairs_2 = non_censored_i & ~non_censored_j & (true_times_j >= true_times_i)
        correct_pairs_2 = (predicted_times_j > predicted_times_i).float()
        
        # Condition 3: i is censored, j is non-censored & i's time is >= j's time
        valid_pairs_3 = censored_i & non_censored_j & (true_times_i >= true_times_j)
        correct_pairs_3 = (predicted_times_i > predicted_times_j).float()
        
        # Compute total valid pairs for each `i`
        total_pairs = valid_pairs_1.float().sum(dim=1) + valid_pairs_2.float().sum(dim=1) + valid_pairs_3.float().sum(dim=1)
        
        # Compute total correct pairs for each `i`
        correct_pairs = (valid_pairs_1 * correct_pairs_1).sum(dim=1) + (valid_pairs_2 * correct_pairs_2).sum(dim=1) + (valid_pairs_3 * correct_pairs_3).sum(dim=1)
        
        # Compute sample losses
        sample_losses = torch.where(total_pairs > 0, correct_pairs / total_pairs, torch.zeros_like(total_pairs))

    
        signals.append(sample_losses.reshape(-1, 1))
        
    signals = np.concatenate(signals, axis = 1)
    np.save(
        f"{log_dir}/cox_cindex_signals.npy",
        signals,
    )
    print("Signals saved to disk.")
    return signals
            

In [85]:
# performance optimization using vectorization
# one model one dataset

def get_cox_model_signals(model_list, data_list, log_dir=global_log_dir):
    print("computing signals using individual c-index...")
    signals = []
    model_idx = 0
    
    for model in model_list:
        print(f"Compute attack signals for model {model_idx}...")
        
        # Convert data to PyTorch tensors
        predicted_times = torch.tensor(model.predict_expectation(data_list[model_idx]).values)  
        true_events = torch.tensor(data_list[model_idx]['event'].values)
        true_times = torch.tensor(data_list[model_idx]['time'].values)

        model_idx += 1
        
        # Get total number of samples
        n = len(predicted_times)
        
        # Expand dimensions to enable broadcasting
        true_times_i = true_times.view(n, 1)
        true_times_j = true_times.view(1, n)
        predicted_times_i = predicted_times.view(n, 1)
        predicted_times_j = predicted_times.view(1, n)
        true_events_i = true_events.view(n, 1)
        true_events_j = true_events.view(1, n)
        
        # Compute masks for different comparison cases
        non_censored_i = true_events_i == 1
        non_censored_j = true_events_j == 1
        censored_i = ~non_censored_i
        
        # Condition 1: Both are non-censored
        valid_pairs_1 = non_censored_i & non_censored_j
        correct_pairs_1 = ((true_times_i > true_times_j) == (predicted_times_i > predicted_times_j)).float()
        correct_pairs_1 += ((true_times_i == true_times_j) & (predicted_times_i == predicted_times_j)).float() * 0.5
        
        # Condition 2: i is non-censored, j is censored & j's time is >= i's time
        valid_pairs_2 = non_censored_i & ~non_censored_j & (true_times_j >= true_times_i)
        correct_pairs_2 = (predicted_times_j > predicted_times_i).float()
        
        # Condition 3: i is censored, j is non-censored & i's time is >= j's time
        valid_pairs_3 = censored_i & non_censored_j & (true_times_i >= true_times_j)
        correct_pairs_3 = (predicted_times_i > predicted_times_j).float()
        
        # Compute total valid pairs for each `i`
        total_pairs = valid_pairs_1.float().sum(dim=1) + valid_pairs_2.float().sum(dim=1) + valid_pairs_3.float().sum(dim=1)
        
        # Compute total correct pairs for each `i`
        correct_pairs = (valid_pairs_1 * correct_pairs_1).sum(dim=1) + (valid_pairs_2 * correct_pairs_2).sum(dim=1) + (valid_pairs_3 * correct_pairs_3).sum(dim=1)
        
        # Compute sample losses
        sample_losses = torch.where(total_pairs > 0, correct_pairs / total_pairs, torch.zeros_like(total_pairs))

    
        signals.append(sample_losses.reshape(-1, 1))
        
    signals = np.concatenate(signals, axis = 1)
    np.save(
        f"{log_dir}/cox_cindex_signals.npy",
        signals,
    )
    print("Signals saved to disk.")
    return signals
            

## audit model using attack signals (pseudo "loss" values)

In [6]:
def run_cox_loss(target_signals: np.ndarray) -> np.ndarray:
    """
    Attack a target model using the LOSS attack.

    Args:
        target_signals (np.ndarray): Softmax value of all samples in the target model.

    Returns:
        np.ndarray: MIA score for all samples (a larger score indicates higher chance of being member). # reverse: larger score -> not member
    """
    #mia_scores = -target_signals  # for cindex signals ???
    mia_scores = -target_signals  # for square signals
    # for square with offsets, mia_scores = target_signals
    return mia_scores

def compute_cox_attack_results(mia_scores, target_memberships):
    """
    Compute attack results (TPR-FPR curve, AUC, etc.) based on MIA scores and membership of samples.

    Args:
        mia_scores (np.array): MIA score computed by the attack.
        target_memberships (np.array): Membership of samples in the training set of target model.

    Returns:
        dict: Dictionary of results, including fpr and tpr list, AUC, TPR at 1%, 0.1% and 0% FPR.
    """
    fpr_list, tpr_list, _ = roc_curve(target_memberships.ravel(), mia_scores.ravel())
    roc_auc = auc(fpr_list, tpr_list)
    one_fpr = tpr_list[np.where(fpr_list <= 0.01)[0][-1]]
    one_tenth_fpr = tpr_list[np.where(fpr_list <= 0.001)[0][-1]]
    zero_fpr = tpr_list[np.where(fpr_list <= 0.0)[0][-1]]

    return {
        "fpr": fpr_list,
        "tpr": tpr_list,
        "auc": roc_auc,
        "one_fpr": one_fpr,
        "one_tenth_fpr": one_tenth_fpr,
        "zero_fpr": zero_fpr,
    }

def get_cox_audit_results(report_dir, model_idx, mia_scores, target_memberships):
    """
    Generate and save ROC plots for attacking a single model.

    Args:
        report_dir (str): Folder for saving the ROC plots.
        model_idx (int): Index of model subjected to the attack.
        mia_scores (np.array): MIA score computed by the attack.
        target_memberships (np.array): Membership of samples in the training set of target model.
        logger (logging.Logger): Logger object for the current run.

    Returns:
        dict: Dictionary of results, including fpr and tpr list, AUC, TPR at 1%, 0.1% and 0% FPR.
    """
    attack_result = compute_cox_attack_results(mia_scores, target_memberships)
    Path(report_dir).mkdir(parents=True, exist_ok=True)

    print(
        f"Target Model {model_idx}: AUC {attack_result['auc']:.4f}, "
        #f"TPR@0.1%FPR {attack_result['one_tenth_fpr']:.4f}, "
        #f"TPR@0.0%FPR {attack_result['zero_fpr']:.4f}"
    )

    plot_roc(
        attack_result["fpr"],
        attack_result["tpr"],
        attack_result["auc"],
        f"{report_dir}/ROC_{model_idx}.png",
    )
    plot_roc_log(
        attack_result["fpr"],
        attack_result["tpr"],
        attack_result["auc"],
        f"{report_dir}/ROC_log_{model_idx}.png",
    )

    np.savez(
        f"{report_dir}/attack_result_{model_idx}",
        fpr=attack_result["fpr"],
        tpr=attack_result["tpr"],
        auc=attack_result["auc"],
        one_tenth_fpr=attack_result["one_tenth_fpr"],
        zero_fpr=attack_result["zero_fpr"],
        scores=mia_scores.ravel(),
        memberships=target_memberships.ravel(),
    )
    return attack_result

def audit_cox_models(
    target_model_indices,
    all_signals,
    all_memberships,
    report_dir=global_log_dir
):
    """
    Audit target model(s) using a Membership Inference Attack algorithm.

    Args:
        report_dir (str): Folder to save attack result.
        target_model_indices (list): List of the target model indices.
        all_signals (np.array): Signal value of all samples in all models (target and reference models).
        all_memberships (np.array): Membership matrix for all models.
        num_reference_models (int): Number of reference models used for performing the attack.
        logger (logging.Logger): Logger object for the current run.
        configs (dict): Configs provided by the user.

    Returns:
        list: List of MIA score arrays for all audited target models.
        list: List of membership labels for all target models.
    """
    all_memberships = np.transpose(all_memberships)

    mia_score_list = []
    membership_list = []

    for target_model_idx in target_model_indices:
        print(
            f"Auditing the privacy risks of target model {target_model_idx}"
        )
        
        mia_scores = run_cox_loss(all_signals[:, target_model_idx])
        target_memberships = all_memberships[:, target_model_idx]

        mia_score_list.append(mia_scores.copy())
        membership_list.append(target_memberships.copy())

        _ = get_cox_audit_results(
            report_dir, target_model_idx, mia_scores, target_memberships
        )

    return mia_score_list, membership_list

# Synthetic Dataset

## Build Cox model

In [66]:
# Initialize the Cox Proportional Hazards model
cox_model = CoxPHFitter()
# Fit the model to the data
cox_model.fit(data, duration_col='time', event_col='event')
cox_model.print_summary()

<lifelines.CoxPHFitter: fitted with 1000 total observations, 87 right-censored observations>
             duration col = 'time'
                event col = 'event'
      baseline estimation = breslow
   number of observations = 1000
number of events observed = 913
   partial log-likelihood = -5381.66
         time fit was run = 2025-03-26 23:33:11 UTC

---
           coef exp(coef)  se(coef)  coef lower 95%  coef upper 95% exp(coef) lower 95% exp(coef) upper 95%
covariate                                                                                                  
Var1       0.02      1.02      0.03           -0.04            0.09                0.96                1.09
Var2       0.01      1.01      0.03           -0.06            0.07                0.94                1.08
Var3      -0.00      1.00      0.03           -0.07            0.06                0.93                1.07
Var4      -0.07      0.93      0.03           -0.13           -0.00                0.88                1.00

           cmp to     z    p  -log2(p)
covariate                             
Var1         0.00  0.69 0.49      1.02
Var2         0.00  0.23 0.82      0.29
Var3         0.00 -0.14 0.89      0.17
Var4         0.00 -2.05 0.04      4.63
---
Concordance = 0.53
Partial AIC = 10771.33
log-likelihood ratio test = 4.84 on 4 df
-log2(p) of ll-ratio test = 1.72

In [ ]:
## optional

# Make predictions - We can predict the survival function for each individual
# Here, we predict the survival function for the first 5 individuals
print("\nPredicted survival functions for the first 5 individuals:")
survival_functions = cox_model.predict_survival_function(data.iloc[:5])
print(survival_functions)

# Plot the survival functions
plt.figure(figsize=(10, 6))
for i in range(survival_functions.shape[1]):
    plt.step(survival_functions.index, survival_functions.iloc[:, i], where="post", label=f"Individual {i+1}")
plt.title("Predicted Survival Functions")
plt.xlabel("Time (weeks)")
plt.ylabel("Survival Probability")
plt.legend()
plt.show()

# Diversity dataset

### half of auditing dataset is from Plus20 (heterogeity) dataset, the non-member ones

In [15]:
# half training, half heterogeneous
def sample_cox_auditing_dataset(training_indices_list, membership):
    """
    Creates a list of DataFrames where each DataFrame is a copy of auditing_data,
    but with values from data at specified training indices.

    Args:
        training_indices_list (list): List of NumPy arrays, each containing indices to overwrite.
        #data (pd.DataFrame): Training data (source of values to copy).
        #auditing_data (pd.DataFrame): Auditing data (target to modify).
        membership: Additional metadata (returned as-is).

    Returns:
        tuple: (list of modified DataFrames, membership)
    """
    auditing_data_list = []

    for training_indices in training_indices_list:
        # Create a deep copy of auditing_data to avoid modifying the original
        modified_data = auditing_data.copy()
        # Override values at training_indices with data's values
        modified_data.loc[training_indices] = data.loc[training_indices]
        auditing_data_list.append(modified_data)

    return auditing_data_list, membership

## workflow

In [23]:
global_log_dir = "heterogeneity/test1"
# debug_cindex_reverse_Treatment_C_success

In [26]:
# dataset size
size = 1000

# training dataset - Reference

parquet_directory = "heterogeneity/"
parquet_files = glob.glob(os.path.join(parquet_directory, "sim_0_dataset_0.parquet"))

dfs = []
for file in parquet_files:
    df = pd.read_parquet(file, engine='pyarrow')  # Specify the PyArrow engine
    dfs.append(df)
data = pd.concat(dfs, ignore_index=True)

data = data[:size]

print(f"Total number of rows: {len(data)}")
print(data.head())

# auditing dataset - AgeAndSex

parquet_files = glob.glob(os.path.join(parquet_directory, "sim_0_dataset_1.parquet"))

dfs = []
for file in parquet_files:
    df = pd.read_parquet(file, engine='pyarrow')  # Specify the PyArrow engine
    dfs.append(df)
auditing_data = pd.concat(dfs, ignore_index=True)

auditing_data = auditing_data[:size]

print(f"Total number of rows: {len(auditing_data)}")
print(auditing_data.head())

#offset_value = 50
#auditing_data['Age'] = auditing_data['Age'] + offset_value

# reverse
# auditing_data['Sex_F'] = np.where(auditing_data['Sex_F'] == 1., 0., 1.)

#print(auditing_data.head())

Total number of rows: 1000
          time  event      Var1       Var2       Var3       Var4       Var5  \
0   171.545403      1  9.205241  10.494158  11.174067  10.223484  10.436789   
1  3153.462869      0  9.814133  10.880935   9.171691  11.162892   9.578770   
2  4435.945878      1  9.399345   8.542752   9.707274  12.183326   9.643746   
3  3758.519765      0  9.462173   8.975298   7.910029   9.303762  10.095466   
4  3807.571035      1  9.330551  10.828557  10.307193  10.574093  11.664707   

        Var6       Var7       Var8       Var9      Var10  
0  10.662066   9.244651   9.483773   8.180704   9.282667  
1   9.240769   8.676237  10.617030  10.187586  10.075526  
2   9.537431  10.727800   9.376040  10.668420  10.155818  
3  10.039023  10.280191   8.694189  12.053568  10.560713  
4   9.897621  10.037702  10.617402  10.045521  10.588995  
Total number of rows: 1000
           time  event       Var1       Var2       Var3       Var4       Var5  \
0   2448.683797      0  10.550747   

In [27]:
## Split dataset randomly

num_model_pairs = 5
data_splits, memberships, training_indices = split_dataframe_for_training(data, num_model_pairs)

In [28]:
## Train cox models and get trained models and metadata

models, metadata = train_cox_models(
    data_splits=data_splits,
    num_model_pairs=num_model_pairs,
    log_dir=global_log_dir
)

#print(metadata[0])  # Metadata for the first model
#models[0].print_summary()  # Summary of the first model

In [29]:
## prepare auditing dataset and memberships

auditing_dataset, auditing_membership = sample_cox_auditing_dataset(training_indices, memberships)

In [30]:
## compute attack signals
signals = get_cox_model_signals(models, auditing_dataset, global_log_dir)

calculating attack signals using square of distance...
Compute attack signals for model 0...
Compute attack signals for model 1...
Compute attack signals for model 2...
Compute attack signals for model 3...
Compute attack signals for model 4...
Compute attack signals for model 5...
Compute attack signals for model 6...
Compute attack signals for model 7...
Compute attack signals for model 8...
Compute attack signals for model 9...
Signals saved to disk.


In [31]:
num_experiments = num_model_pairs * 2
target_model_indices = list(range(num_experiments))
mia_score_list, membership_list = audit_cox_models(
    target_model_indices,
    signals,
    auditing_membership,
    global_log_dir
)

Auditing the privacy risks of target model 0
Target Model 0: AUC 0.7629, 
Auditing the privacy risks of target model 1
Target Model 1: AUC 0.7436, 
Auditing the privacy risks of target model 2
Target Model 2: AUC 0.7702, 
Auditing the privacy risks of target model 3
Target Model 3: AUC 0.7322, 
Auditing the privacy risks of target model 4
Target Model 4: AUC 0.7659, 
Auditing the privacy risks of target model 5
Target Model 5: AUC 0.7388, 
Auditing the privacy risks of target model 6
Target Model 6: AUC 0.7628, 
Auditing the privacy risks of target model 7
Target Model 7: AUC 0.7458, 
Auditing the privacy risks of target model 8
Target Model 8: AUC 0.7764, 
Auditing the privacy risks of target model 9
Target Model 9: AUC 0.7325, 


<Figure size 640x480 with 0 Axes>